In [47]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


 Because of this is notebook we don't run from Makfile directory so use " getcwd() "

In [10]:
current_dir = Path(os.getcwd())
print(f"the path is {current_dir}")
file = current_dir / '../archive/PRSA_Data_Aotizhongxin_20130301-20170228.csv'

the path is /home/ahmed-sulieman/Desktop/AI_projects/Air_Pol/notebook


# Makefile

run this below code when you run from the Makfile

In [11]:
# current_dir = Path(__file__).parent
# file = current_dir / '../archive/PRSA_Data_Aotizhongxin_20130301-20170228.csv'

In [41]:
if not file.exists():
    print('file not exist')
    raise FileNotFoundError(f"File {file} does not exist.")

In [42]:
df = pd.read_csv(file)

In [43]:
df = df.drop(columns=['No', 'station'])
df_b = df.copy()

In [44]:
df.fillna(method='ffill', inplace=True)
df_b.dropna(inplace=True)

/tmp/ipykernel_16866/490933577.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [55]:
def split_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Split the 'date' column into separate 'year', 'month', 'day', and 'hour' columns.
    Parameters:
    df (pd.DataFrame): The input DataFrame containing a 'date' column.
    Returns:
        pd.DataFrame: A DataFrame with the 'date' column split into separate columns.
    """
    try:
        train_raw = df[df['year'] < 2016].copy()
        test_raw = df[df['year'] >= 2016].copy()
        df_test = test_raw.dropna().copy()
        logging.info("Date column split successfully.")
        return train_raw, df_test
    except Exception as e:
        logging.error(f"Error occurred while splitting date: {e}")
        return df

def get_processed_data(file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load and process the air pollution data from a CSV file.
    Parameters:
    file (str): The name of the CSV file containing the data.
    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: A tuple containing the processed DataFrame 
    with filled missing values and a copy of the DataFrame with dropped missing values.
    """
    try:
        current_dir = Path(os.getcwd())
        file_path = current_dir / '../archive' / file
        # current_dir = Path(__file__).parent
        # file_path = current_dir / '../archive' / file

        if not file_path.exists():
            logging.error(f"File {file_path} does not exist.")
            print('file not exist')
            return None, None, None
        df = pd.read_csv(file_path)
        df = df.drop(columns=['No', 'station'])

        df_train, df_test = split_dataset(df)
        
        if df_train is None or df_test is None:
            logging.error("Failed to split dataset.")
            return None, None, None
        
        df_dropped_values = df_train.copy()

        df_train.fillna(method='ffill', inplace=True)
        df_dropped_values.dropna(inplace=True)

        logging.info("Data loaded and processed successfully.")
        return df_train, df_dropped_values, df_test
    except Exception as e:
        logging.error(f"Error occurred: {e}")
        return None, None, None

    

In [56]:
file = 'PRSA_Data_Aotizhongxin_20130301-20170228.csv'
df, df_b, df_test = get_processed_data(file)
print(f"df shape: {df.shape} | df_b shape: {df_b.shape} | df_test shape: {df_test.shape}")

2026-05-08 21:36:55,771 - INFO - Date column split successfully.
/tmp/ipykernel_16866/4005916918.py:49: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_train.fillna(method='ffill', inplace=True)
2026-05-08 21:36:55,800 - INFO - Data loaded and processed successfully.


df shape: (24864, 16) | df_b shape: (22235, 16) | df_test shape: (9580, 16)
